In [ ]:
# Path to the reference dataset repository (set before running this notebook)
OG_REPO_ROOT = '<path/to/reference/repository>'

import pandas as pd
from logging import getLogger, StreamHandler
import sys

logger = getLogger(__name__)
logger.addHandler(StreamHandler(stream=sys.stdout))

OG_data_path = f'{OG_REPO_ROOT}/data/'
PROD_data_path = 'data/'


def load_and_compare_dfs(dataset_A_path:str , dataset_B_path:str, verbose=False):
    """Low level util to load and call compare"""

    data_A = pd.read_csv(f'{dataset_A_path}')
    data_B = pd.read_csv(f'{dataset_B_path}')

    if verbose:
      print(f'A.shape: {data_A.shape}, B.shape: {data_B.shape}')
      print('A.head():\n', data_A.head(5))
      print('B.head():\n', data_B.head(5))

    comparison = data_A.compare(data_B, result_names=('Original', 'New'))
    return comparison

## datasets

In [18]:

def compare_dataset_with_OG(dataset_name: str, phases = ['raw', 'prefix','prompt']):
  """"""

  phase_results = {}
  for phase in phases:
    print(f'\nComparing {dataset_name} - {phase}')
    comparison = load_and_compare_dfs(
      dataset_A_path=ff'{OG_REPO_ROOT}/data/.cache/{phase}/{dataset_name}.csv',
      dataset_B_path=f'data/.cache/{phase}/{dataset_name}.csv',
      verbose=(phase == 'raw')
    )

    print(f'Comparison shape: {comparison.shape}')
    if comparison.empty:
      print('✅ Datasets are identical.')
    else: # this never happens, compare throws error
      print('❌ Datasets differ. Sample differences:\n', comparison.head())
    phase_results[phase] = comparison
  return phase_results


In [19]:
results = compare_dataset_with_OG('twitter')


Comparing twitter - raw
A.shape: (35921, 1), B.shape: (35921, 1)
A.head():
                                                 text
0  @anonymized_account jeszcze zaraz będzie rady ...
1  Komisja Europejska szantażuje polski rząd? Pol...
2  @anonymized_account Ten kto go zaprosił życzy ...
3  @anonymized_account I się tumanie zdziwisz. Ob...
4  @anonymized_account Był tam też pan, który na ...
B.head():
                                                 text
0  @anonymized_account jeszcze zaraz będzie rady ...
1  Komisja Europejska szantażuje polski rząd? Pol...
2  @anonymized_account Ten kto go zaprosił życzy ...
3  @anonymized_account I się tumanie zdziwisz. Ob...
4  @anonymized_account Był tam też pan, który na ...
Comparison shape: (0, 0)
✅ Datasets are identical.

Comparing twitter - prefix
Comparison shape: (0, 0)
✅ Datasets are identical.

Comparing twitter - prompt
Comparison shape: (0, 0)
✅ Datasets are identical.


In [6]:
results["prefix"]

prefix  \
                                                Original   
0      @user2931 Ten kto go zaprosił życzy dobrze prz...   
1                      @user8364 I się tumanie zdziwisz.   
2      @user7219 Był tam też pan, który na pytania o ...   
3      @user2537 Wielu z nas wiedziało już od wielu l...   
5      @user1034 Trzaskowski w czasie wojny nazywał s...   
...                                                  ...   
17636  @user6082 Pierwszy sezon zapowiada się na powo...   
17638  @user6741 Morawiecki niech sobie w d...e wsadz...   
17639  @user8344 Marysiu, ludzie będą umierać na potęgę.   
17644                   @user9439 To nic nie da siostry.   
17645      Poseł @user1794 obchodzi dzisiaj 58 urodziny!   

                                                          
                                                     New  
0      @user2511 Ten kto go zaprosił życzy dobrze prz...  
1                      @user9039 I się tumanie zdziwisz.  
2      @user7302 Był tam też pan, który na pytania o ...  
3      @user3880 Wielu z nas wiedziało już od wielu l...  
5      @user6352 Trzaskowski w czasie wojny nazywał s...  
...                                                  ...  
17636  @user4445 Pierwszy sezon zapowiada się na powo...  
17638  @user1086 Morawiecki niech sobie w d...e wsadz...  
17639  @user9982 Marysiu, ludzie będą umierać na potęgę.  
17644                   @user1497 To nic nie da siostry.  
17645      Poseł @user4596 obchodzi dzisiaj 58 urodziny!  

[9662 rows x 2 columns]

In [ ]:
import re

def normalize_ats(text):
    """Replace all @userXXXX with a canonical placeholder for comparison."""
    return re.sub(r'@user\d+', '@user####', str(text))

def review_diffs_side_by_side(dataset_name: str, phase: str = 'prefix', n: int = 20):
    """
    Show differing rows side by side.
    After normalization of @userXXXX, check if any real content differences remain.
    """
    og  = pd.read_csv(ff'{OG_REPO_ROOT}/data/.cache/{phase}/{dataset_name}.csv')
    prod = pd.read_csv(f'data/.cache/{phase}/{dataset_name}.csv')

    col = og.columns[0]
    diff_idx = og.index[og[col] != prod[col]]
    print(f'{dataset_name} / {phase}: {len(diff_idx)} differing rows')

    # Check how many differ even after normalizing @userXXXX
    still_different = [
        i for i in diff_idx
        if normalize_ats(og.loc[i, col]) != normalize_ats(prod.loc[i, col])
    ]
    print(f'  Still different after @user normalization: {len(still_different)}')
    if still_different:
        print('  *** Non-AT differences found! Showing first 5: ***')
        for i in still_different[:5]:
            print(f'  [{i}] OG:   {og.loc[i, col]}')
            print(f'  [{i}] prod: {prod.loc[i, col]}')
            print()

    print(f'\nSample side-by-side (first {n} diffs):')
    print(f"{'IDX':>6}  {'OG':^60}  {'PROD':^60}")
    print('-' * 130)
    for i in list(diff_idx)[:n]:
        og_val   = str(og.loc[i, col])[:58]
        prod_val = str(prod.loc[i, col])[:58]
        print(f'{i:>6}  {og_val:<60}  {prod_val:<60}')

review_diffs_side_by_side('twitter', 'prefix')

##  domains

In [22]:
def compare_domains_to_OG(domain_names: list):
  """"""
  domain_results = {}
  for domain_name in domain_names:
    print(f'\nComparing {domain_name} domain')

    comparison = load_and_compare_dfs(
      dataset_A_path=ff'{OG_REPO_ROOT}/data/preprocessed/domains/{domain_name}.csv',
      dataset_B_path=f'data/preprocessed/domains/{domain_name}.csv',
      verbose=True
    )

    print(f'Comparison shape: {comparison.shape}')
    if comparison.empty:
      print('✅ Domain data is identical.')
    else:
      print('❌ Domain data differ. Sample differences:\n', comparison.head())
    domain_results[domain_name] = comparison
  return domain_results

In [21]:
# domains = ['wiki', 'social', 'reviews', 'lit']
domains = ['social']
domain_composition = compare_domains_to_OG(domains)


Comparing social domain
A.shape: (12000, 5), B.shape: (12000, 5)
A.head():
                                                 text  \
0  Moje ciało jest moją twierdzą ale jest miejsce...   
1  @user9882 Doskonale to rozumiem, PiS nie radzi...   
2  @user3403 Musimy zadbać żeby tego błędu więcej...   
3  @user6418 Jestem na emeryturze i stać mnie na ...   
4  @user6574 Briefing posła Budki w styczniu br. ...   

                                              prefix  \
0  Moje ciało jest moją twierdzą ale jest miejsce...   
1                   @user9882 Doskonale to rozumiem,   
2  @user3403 Musimy zadbać żeby tego błędu więcej...   
3  @user6418 Jestem na emeryturze i stać mnie na ...   
4      @user6574 Briefing posła Budki w styczniu br.   

                                              prompt  dataset  domain  
0  Dokończ następujący post, zachowując jego styl...  twitter  social  
1  Dokończ następujący post, zachowując jego styl...  twitter  social  
2  Dokończ następujący post, zach

In [13]:
old_social = pd.read_csv(f'{OG_REPO_ROOT}/data/preprocessed/domains/social.csv')
new_social = pd.read_csv('data/preprocessed/domains/social.csv')

In [15]:
old_social.head()

,text,prefix,prompt,dataset,domain
0,Moje ciało jest moją twierdzą ale jest miejsce...,Moje ciało jest moją twierdzą ale jest miejsce...,"Dokończ następujący post, zachowując jego styl...",twitter,social
1,"@user9882 Doskonale to rozumiem, PiS nie radzi...","@user9882 Doskonale to rozumiem,","Dokończ następujący post, zachowując jego styl...",twitter,social
2,@user3403 Musimy zadbać żeby tego błędu więcej...,@user3403 Musimy zadbać żeby tego błędu więcej...,"Dokończ następujący post, zachowując jego styl...",twitter,social
3,@user6418 Jestem na emeryturze i stać mnie na ...,@user6418 Jestem na emeryturze i stać mnie na ...,"Dokończ następujący post, zachowując jego styl...",twitter,social
4,@user6574 Briefing posła Budki w styczniu br. ...,@user6574 Briefing posła Budki w styczniu br.,"Dokończ następujący post, zachowując jego styl...",twitter,social


In [16]:
new_social.head()

,text,prefix,prompt,dataset,domain
0,Moje ciało jest moją twierdzą ale jest miejsce...,Moje ciało jest moją twierdzą ale jest miejsce...,"Dokończ następujący post, zachowując jego styl...",twitter,social
1,"@user3198 Doskonale to rozumiem, PiS nie radzi...","@user3198 Doskonale to rozumiem,","Dokończ następujący post, zachowując jego styl...",twitter,social
2,@user4386 Musimy zadbać żeby tego błędu więcej...,@user4386 Musimy zadbać żeby tego błędu więcej...,"Dokończ następujący post, zachowując jego styl...",twitter,social
3,@user8674 Jestem na emeryturze i stać mnie na ...,@user8674 Jestem na emeryturze i stać mnie na ...,"Dokończ następujący post, zachowując jego styl...",twitter,social
4,@user6359 Briefing posła Budki w styczniu br. ...,@user6359 Briefing posła Budki w styczniu br.,"Dokończ następujący post, zachowując jego styl...",twitter,social


In [10]:
domain_composition["social"]

text  \
                                                Original   
1      @user9882 Doskonale to rozumiem, PiS nie radzi...   
2      @user3403 Musimy zadbać żeby tego błędu więcej...   
3      @user6418 Jestem na emeryturze i stać mnie na ...   
4      @user6574 Briefing posła Budki w styczniu br. ...   
6      Wraz z @user5618 i liderami grup politycznych ...   
...                                                  ...   
11995  @user9071: No pewnie było by lepiej dla przeds...   
11996  @user4943: ju nawet nie bede podawal argumento...   
11997  @user4343: Palancie, zdanie "coraz więcej kobi...   
11998  @user4034: zadzwoń do Sztabu Generalnego w Kij...   
11999  Stojąc na trójnogu, czyli znacząc tyle samo il...   

                                                          \
                                                     New   
1      @user3198 Doskonale to rozumiem, PiS nie radzi...   
2      @user4386 Musimy zadbać żeby tego błędu więcej...   
3      @user8674 Jestem na emeryturze i stać mnie na ...   
4      @user6359 Briefing posła Budki w styczniu br. ...   
6      Wraz z @user2385 i liderami grup politycznych ...   
...                                                  ...   
11995  @user5106: No pewnie było by lepiej dla przeds...   
11996  @user8174: ju nawet nie bede podawal argumento...   
11997  @user8285: Palancie, zdanie "coraz więcej kobi...   
11998  @user6822: zadzwoń do Sztabu Generalnego w Kij...   
11999  Stojąc na trójnogu, czyli znacząc tyle samo il...   

                                                  prefix  \
                                                Original   
1                       @user9882 Doskonale to rozumiem,   
2      @user3403 Musimy zadbać żeby tego błędu więcej...   
3      @user6418 Jestem na emeryturze i stać mnie na ...   
4          @user6574 Briefing posła Budki w styczniu br.   
6      Wraz z @user5618 i liderami grup politycznych ...   
...                                                  ...   
11995  @user9071: No pewnie było by lepiej dla przeds...   
11996  @user4943: ju nawet nie bede podawal argumento...   
11997  @user4343: Palancie, zdanie "coraz więcej kobi...   
11998  @user4034: zadzwoń do Sztabu Generalnego w Kij...   
11999                                                NaN   

                                                          \
                                                     New   
1                       @user3198 Doskonale to rozumiem,   
2      @user4386 Musimy zadbać żeby tego błędu więcej...   
3      @user8674 Jestem na emeryturze i stać mnie na ...   
4          @user6359 Briefing posła Budki w styczniu br.   
6      Wraz z @user2385 i liderami grup politycznych ...   
...                                                  ...   
11995  @user5106: No pewnie było by lepiej dla przeds...   
11996  @user8174: ju nawet nie bede podawal argumento...   
11997  @user8285: Palancie, zdanie "coraz więcej kobi...   
11998  @user6822: zadzwoń do Sztabu Generalnego w Kij...   
11999                                                NaN   

                                                  prompt  \
                                                Original   
1      Dokończ następujący post, zachowując jego styl...   
2      Dokończ następujący post, zachowując jego styl...   
3      Dokończ następujący post, zachowując jego styl...   
4      Dokończ następujący post, zachowując jego styl...   
6      Dokończ następujący post, zachowując jego styl...   
...                                                  ...   
11995  Dokończ następujący post, zachowując jego styl...   
11996  Dokończ następujący post, zachowując jego styl...   
11997  Dokończ następujący post, zachowując jego styl...   
11998  Dokończ następujący post, zachowując jego styl...   
11999                                                NaN   

                                                          
                                                     New  
1      Dokończ następujący po

In [ ]:
all_datasets = [
  'wiki',
  'plsc',
  'coursebooks',
  'classics',
  'twitter', 
  'wykop',
  'polemo_hotels',
  'polemo_medicine',
  'polemo_products',
  'polemo_courses',
  'allegro',
  'filmweb',
  'pmrd',
  'wikinews',
  'gov'             
]

In [ ]:
def get_column_name(phase):
    return {
        'raw': 'text',
        'prefix': 'prefix',
        'prompt': 'prompt'
    }.get(phase, f"Unknown phase: {phase}")

def compare_by_text(data_a, data_b):
    return data_a.compare(data_b, result_names=('Original', 'New'))

## `generation_input.csv`

In [25]:
# check length of OG's one


comparison = load_and_compare_dfs(
      dataset_A_path=f'{OG_REPO_ROOT}/generation_input.csv',
      dataset_B_path='generation_input.csv',
)

if comparison.empty:
    print('✅ Generation input datasets are identical.')
else:
    print('❌ Generation input datasets differ. Sample differences:\n', comparison.head())

✅ Generation input datasets are identical.


## postprocessing

In [4]:
import filecmp
import pathlib

def compare_postprocessing(
    prod_path='data/postprocessed',
    og_path=f'{OG_REPO_ROOT}/data/prod/deterministic_split_final_test',
    splits=('training', 'test', 'dev'),
    exts=('txt', 'key', 'meta'),
    show_sample_diffs=True,
    n_sample=3,
):
    """Compare postprocessed outputs split-by-split against an OG reference.

    Prints a table of line counts and identity status for each
    split × extension combination, then shows a sample of the first
    differing lines when files don't match.

    Returns True if every checked file is byte-for-byte identical.
    """
    prod_path = pathlib.Path(prod_path)
    og_path   = pathlib.Path(og_path)
    all_match = True

    for split in splits:
        print(f"\n── {split} ──")
        for ext in exts:
            prod_file = prod_path / split / f'start.{ext}'
            og_file   = og_path   / split / f'start.{ext}'

            if not prod_file.exists():
                print(f"  start.{ext}:  ❌  MISSING in prod")
                all_match = False
                continue
            if not og_file.exists():
                print(f"  start.{ext}:  ⚠️   MISSING in OG (skipped)")
                continue

            og_bytes   = og_file.read_bytes()
            prod_bytes = prod_file.read_bytes()
            og_n   = og_bytes.count(b'\n')
            prod_n = prod_bytes.count(b'\n')

            # filecmp.cmp does a full byte comparison when shallow=False
            identical = filecmp.cmp(str(prod_file), str(og_file), shallow=False)

            if identical:
                print(f"  start.{ext}:  ✅  IDENTICAL  ({prod_n} lines)")
            else:
                all_match = False
                count_note = '' if prod_n == og_n else f', OG={og_n} vs prod={prod_n}'
                print(f"  start.{ext}:  ❌  DIFFERS{count_note}")

                if show_sample_diffs and og_n > 0:
                    og_lines   = og_file.read_text(errors='replace').splitlines()
                    prod_lines = prod_file.read_text(errors='replace').splitlines()
                    diffs = [
                        (i, og_lines[i], prod_lines[i])
                        for i in range(min(len(og_lines), len(prod_lines)))
                        if og_lines[i] != prod_lines[i]
                    ]
                    for idx, og_val, prod_val in diffs[:n_sample]:
                        print(f"    line {idx+1}:")
                        print(f"      OG:   {og_val[:120]!r}")
                        print(f"      prod: {prod_val[:120]!r}")
                    if len(diffs) > n_sample:
                        print(f"    … and {len(diffs) - n_sample} more differing lines")

    print(f"\n{'🎉 All files match!' if all_match else '⚠️  Some files differ — see above.'}")
    return all_match


### alpha

In [ ]:

alpha = {
    'prod_path': 'data/postprocessed',
    'og_path': f'{OG_REPO_ROOT}/data/prod/deterministic_split_final_test',
    'splits': ('training', 'test', 'dev'),
    'exts': ('txt', 'key', 'meta'),
    'show_sample_diffs': True,
    'n_sample': 3,
} # defaults

compare_postprocessing(**alpha) # defaault values


── training ──
  start.txt:  ✅  IDENTICAL  (35763 lines)
  start.key:  ✅  IDENTICAL  (35763 lines)
  start.meta:  ✅  IDENTICAL  (35763 lines)

── test ──
  start.txt:  ✅  IDENTICAL  (4505 lines)
  start.key:  ✅  IDENTICAL  (4505 lines)
  start.meta:  ✅  IDENTICAL  (4505 lines)

── dev ──
  start.txt:  ❌  DIFFERS, OG=0 vs prod=4481
  start.key:  ✅  IDENTICAL  (4481 lines)
  start.meta:  ✅  IDENTICAL  (4481 lines)

⚠️  Some files differ — see above.


False

### beta

In [8]:
beta = {
    'prod_path': 'data/postprocessed_beta',
    'og_path': f'{OG_REPO_ROOT}/data/prod/beta',
    'splits': ('',),   # beta has no subdirectory split — files are at the path root
}

compare_postprocessing(**beta)


──  ──
  start.txt:  ✅  IDENTICAL  (4356 lines)
  start.key:  ✅  IDENTICAL  (4356 lines)
  start.meta:  ✅  IDENTICAL  (4356 lines)

🎉 All files match!


True

### gamma

In [9]:
gamma = {
    'prod_path': 'data/postprocessed_gamma',
    'og_path': f'{OG_REPO_ROOT}/data/prod/gamma',
    'splits': ('',),   # gamma has no subdirectory split — files are at the path root
}

compare_postprocessing(**gamma)


──  ──
  start.txt:  ✅  IDENTICAL  (19914 lines)
  start.key:  ✅  IDENTICAL  (19914 lines)
  start.meta:  ✅  IDENTICAL  (19914 lines)

🎉 All files match!


True

### forging test A and B

In [8]:
testA = {
    'prod_path': 'data/test_A',
    'og_path': f'{OG_REPO_ROOT}/data/prod/test_A',
    'splits': ('',),
}

testB = {
    'prod_path': 'data/test_B',
    'og_path': f'{OG_REPO_ROOT}/data/prod/test_B',
    'splits': ('',),
}

compare_postprocessing(**testA)
compare_postprocessing(**testB)


──  ──
  start.txt:  ✅  IDENTICAL  (10343 lines)
  start.key:  ✅  IDENTICAL  (10343 lines)
  start.meta:  ✅  IDENTICAL  (10343 lines)

🎉 All files match!

──  ──
  start.txt:  ✅  IDENTICAL  (18432 lines)
  start.key:  ✅  IDENTICAL  (18432 lines)
  start.meta:  ✅  IDENTICAL  (18432 lines)

🎉 All files match!


True